# Reaction cycle for P1

In [2]:
import sys
import subprocess
import importlib.util
# Check if symfit is already installed
if importlib.util.find_spec("symfit") is None:
    print("Installing symfit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "symfit"])
    print("SymFit installed successfully!")
else:
    print("SymFit is already installed.")

# Import symfit and verify installation
import symfit
print(f"SymFit version: {symfit.__version__}")

import pandas as pd
import numpy as np
from symfit import parameters,variables,Fit,D,ODEModel
import matplotlib.pyplot as plt

dat = pd.read_csv('0p1mM-LP1_1-eq-MAD_50-eq-AdaP_10-eq-Dithiane.csv', \
                  sep=',')
dat.head()

SymFit is already installed.
SymFit version: 0.5.6


,hrs,LP-fr,MP-fr,CP-fr,MPM-fr,LP,MP,CP,MPM
0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000100,0.000000,0.000000e+00,0.000000e+00
1,0.050000,0.977828,0.020374,0.001545,0.000253,0.000098,0.000002,1.544890e-07,2.532000e-08
2,0.566667,0.768548,0.190858,0.039106,0.001488,0.000077,0.000019,3.910590e-06,1.488300e-07
3,1.183333,0.676951,0.229782,0.092666,0.000533,0.000068,0.000023,9.266610e-06,5.329270e-08
4,1.783333,0.650274,0.212720,0.136589,0.000201,0.000065,0.000021,1.365890e-05,2.012520e-08


In [3]:
t_data=np.array(dat.hrs, dtype=np.float64)
p_data=np.array(dat.LP, dtype=np.float64)
pm_data=np.array(dat.MP, dtype=np.float64)
cp_data=np.array(dat.CP, dtype=np.float64)
ol_data=np.array(dat.MPM, dtype=np.float64)

In [4]:
#import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import fixed
from IPython.display import display

%matplotlib nbagg
%matplotlib inline 
plt.figure()

def update_plot(k1, k2, k3, k5, k6, k7, k8, A, E, J, M, time):

    # define reaction rate constants
    k9 = 0.00001
    k10 = 2000
    

    # initial condition
    C0 = [A, 0, 0, 0, 0, 0, E, J, 0, 0, M]
    
    # function that returns Cout
    def model(C,t):
        A = C[0]
        B = C[1]
        O = C[2]
        S = C[3]
        #F = C[4]
        H = C[4]
        K = C[5]
        E = C[6]
        J = C[7]
        I = C[8]
        G = C[9]
        M = C[10]
        dAdt = -k1*A*M + k6*H
        dBdt = k1*A*M - k2*B - k3*B*M - k7*B*G
        dOdt = k3*B*M - k10*O*G*G
        dSdt = k2*B - k5*S*G #+ k4a*F - k4*S
        #dFdt = k4*S - k4a*F
        dHdt = k7*B*G + k5*S*G + k10*O*G*G - k6*H
        dKdt = k6*H + k10*O*G*G
        dEdt = -k8*E*J
        dJdt = -k8*E*J - k9*J
        dIdt = k8*E*J + k9*J
        dGdt = k8*E*J - k7*B*G - k5*S*G - 2*k10*O*G*G
        dMdt = -k1*A*M - k3*B*M
        Cout = [dAdt, dBdt, dOdt, dSdt, dHdt, dKdt, dEdt, dJdt, dIdt, dGdt, dMdt]
        return Cout


    # time points
    steps = 10000
    t = np.linspace(0,time,steps)

    # solve ODE
    Cint = odeint(model,C0,t)

    # simple plot results
    plt.figure(figsize=(10, 8))
    plt.clf() # clear the plot
    plt.plot(t/3600,Cint[:,0]*1000,'-', color='black', label=r'LP')
    plt.plot(t/3600,Cint[:,1]*1000,'-', color='red', label=r'MP')
    plt.plot(t/3600,Cint[:,2]*1000,'-', color='green', label=r'MPM')
    plt.plot(t/3600,Cint[:,3]*1000,'-', color='blue', label=r'CP')
    #plt.plot(t,Cint[:,4],'--',label=r'CPN')
    #plt.plot(t/3600,Cint[:,4]*1000,'-', color='purple', label=r'MP-DTT')
    
    plt.scatter(t_data, p_data*1000, marker='s', s=120, color='black', label='LP')
    plt.scatter(t_data, pm_data*1000, marker='o', s=120, color='red', label='MP')
    plt.scatter(t_data, ol_data*1000, marker='v', s=120, color='green', label='MPM')
    plt.scatter(t_data, cp_data*1000, marker='^', s=120, color='blue', label='CP')

    plt.ylabel('Concentration (mM)',fontsize=28)
    plt.xlabel('Time (h)', fontsize=28)
    plt.xticks(fontsize=26, fontweight = 'normal')
    plt.yticks(fontsize=26, fontweight = 'normal')
    plt.legend(loc='best')
    plt.show()

k1 = widgets.FloatSlider(min=0.01, max=100, value=3, step=0.01, description='k1')    
k2 = widgets.FloatSlider(min=0.00001, max=100, value=0.00015, step=0.00001, description='k2', readout_format='.5f')    
k3 = widgets.FloatSlider(min=0.00001, max=200, value=0.1, step=0.00001, description='k3', readout_format='.5f')
#k4 = widgets.FloatSlider(min=0.00001, max=200, value=1, step=0.00001, description='k4', readout_format='.5f')
#k4a = widgets.FloatSlider(min=0.00001, max=200, value=0.5, step=0.00001, description='k4a', readout_format='.5f')
k5 = widgets.FloatSlider(min=0.00001, max=200, value=0.008, step=0.00001, description='k5', readout_format='.5f')
k6 = widgets.FloatSlider(min=0.00001, max=200, value=0.002, step=0.00001, description='k6', readout_format='.5f')
k7 = widgets.FloatSlider(min=0.0, max=2000, value=100, step=0.00001, description='k7', readout_format='.5f')
k8 = widgets.FloatSlider(min=0.00001, max=200, value=0.00105, step=0.00001, description='k8', readout_format='.5f')
#k10 = widgets.FloatSlider(min=0.00001, max=200, value=0.0001, step=0.00001, description='k10', readout_format='.5f')
A = widgets.FloatSlider(min=0, max=0.01, value=0.0001, step=0.0001, description='LP', readout_format='.5f')
E = widgets.FloatSlider(min=0, max=0.1, value=0.001, step=0.0001, description='Dith', readout_format='.5f')
J = widgets.FloatSlider(min=0, max=0.1, value=0.005, step=0.0001, description='AdaP', readout_format='.5f')
M = widgets.FloatSlider(min=0, max=0.1, value=0.0001, step=0.0001, description='MAD', readout_format='.5f')
time = widgets.FloatSlider(min=1, max=650000, value=300000, step=1800, description='time')
widgets.interact(update_plot, k1=k1, k2=k2, k3=k3, k5=k5, k6=k6, k7=k7, k8=k8, A=A, E=E, J=J, M=M, time=time)
 

interactive(children=(FloatSlider(value=3.0, description='k1', min=0.01, step=0.01), FloatSlider(value=0.00015…

<function __main__.update_plot(k1, k2, k3, k5, k6, k7, k8, A, E, J, M, time)>